### ЗАДАЧА: Умные локеры для хранения заказов

Сервис доставки использует сухие и холодильные локеры для краткого хранения заказов.
Нужно собрать систему, которая:
- принимает корректные посылки,
- отбрасывает неправильные или конфликтующие записи,
- контролирует совместимость режима хранения,
- следит за занятым объёмом в каждом локере,
- показывает, какой локер заполнен сильнее всего.

В части строк указан неизвестный локер,
в части объём заполнен неверно,
часть посылок не подходит по режиму хранения,
а некоторые записи дублируют уже обработанный `parcel_id`.
        


In [7]:
from dataclasses import dataclass
from typing import Optional


locker_specs = {
    'COLD-1': {'capacity_l': 8, 'mode': 'cold'},
    'COLD-2': {'capacity_l': 6, 'mode': 'cold'},
    'DRY-1': {'capacity_l': 10, 'mode': 'dry'},
}

# rows: parcel_id|locker_id|client|volume_l|mode
rows = [
    'PX-100|COLD-1|Alice|2|cold',
    'PX-101|DRY-1|Bob|3|dry',
    'PX-102|COLD-1|Kira|5|dry',
    'PX-103|HOT-9|Max|1|dry',
    'PX-104|COLD-2|Eva|0|cold',
    'PX-101|DRY-1|Bob|2|dry',
    'PX-105|COLD-1|Lena|7|cold',
    'PX-106|COLD-2|Ira|4|cold',
]


class LockerError(Exception):
    pass


class RowFormatError(LockerError):
    pass


class LockerNotFoundError(LockerError):
    pass


class VolumeError(LockerError):
    pass


class StorageModeError(LockerError):
    pass


class CapacityLimitError(LockerError):
    pass


class DuplicateParcelError(LockerError):
    pass


@dataclass(order=True)
class Parcel:
    volume_l: int
    parcel_id: str
    locker_id: str
    client: str
    mode: str


class Locker:
    def __init__(self, locker_id, capacity_l, mode):
        # TODO: сохранить locker_id, capacity_l и mode
        # TODO: создать список parcels
        self.locker_id = locker_id
        self.capacity_l = capacity_l
        self.mode = mode
        self.parcels =[]
        

    def has_parcel(self, parcel_id):
        # TODO: пройтись по self.parcels
        # TODO: вернуть True, если parcel.parcel_id совпал
        # TODO: иначе вернуть False
        for el in self.parcels:
            if el.parcel_id == parcel_id:
                return True
        return False

    def used_volume(self):
        # TODO: вернуть сумму volume_l по всем parcel в self.parcels
        c = 0 
        for e in self.parcels:
            c += e.volume_l
        return c

    def free_volume(self):
        # TODO: вернуть capacity_l - used_volume()
        return self.capacity_l - self.used_volume()
        

    def add_parcel(self, parcel):
        # TODO: если parcel.mode не совпадает с self.mode -> raise StorageModeError(...)
        # TODO: если has_parcel(parcel.parcel_id) -> raise DuplicateParcelError(...)
        # TODO: если parcel.volume_l > free_volume() -> raise CapacityLimitError(...)
        # TODO: добавить посылку в self.parcels
        # TODO: отсортировать self.parcels
        if self.mode != parcel.mode:
            raise StorageModeError('режимы должны совпадать!')
        if self.has_parcel(parcel.parcel_id):
            raise DuplicateParcelError('такой айди уже есть!')
        if parcel.volume_l > self.free_volume():
            raise CapacityLimitError('volume_l больше free_volume!')
        self.parcels.append(parcel)
        self.parcels.sort()

class LockerService:
    def __init__(self, locker_specs):
        # TODO: создать storage вида locker_id -> Locker(...)
        # TODO: создать списки accepted и errors
        # TODO: создать множество processed_ids для глобальной проверки дубликатов
        self.storage = {}
        for id, params in locker_specs.items():
            self.storage.setdefault(id, Locker(capacity_l=params['capacity_l'], locker_id=id, mode=params['mode']))
        self.accepted = []
        self.errors = []
        self.processed_ids = set()




    def parse_parcel(self, row):
        # TODO: split по '|'
        # TODO: ожидать 5 частей: parcel_id, locker_id, client, volume_raw, mode
        # TODO: если частей не 5 -> raise RowFormatError(...)
        # TODO: проверить, что locker_id существует
        # TODO: volume_raw преобразовать в int
        # TODO: при ошибке преобразования использовать raise VolumeError(...) from exc
        # TODO: если volume_l <= 0 -> raise VolumeError(...)
        # TODO: mode проверить на {'cold', 'dry'}
        # TODO: вернуть объект Parcel(...)
        arr = row.split('|')
        if len(arr) != 5:
            raise RowFormatError('ожидалось 5 частей!')
        parcel_id, locker_id, client, volume_raw, mode = arr[0], arr[1], arr[2], arr[3], arr[4]
        if locker_id not in self.storage.keys():
            raise LockerNotFoundError('не удается найти такой locker_id!')
        try:
            volume_raw = int(volume_raw)
        except LockerError as e:
            raise VolumeError('некорректный volume_raw!') from e
        if volume_raw <= 0:
            raise VolumeError('volume_raw должно быть больше 0!')
        if mode not in {'cold', 'dry'}:
            raise StorageModeError('некорректное значение mode!')
        return Parcel(parcel_id=parcel_id, mode=mode, volume_l=volume_raw, locker_id=locker_id, client=client)

    def submit(self, row):
        # TODO: внутри try вызвать parse_parcel(row)
        # TODO: если parcel.parcel_id уже в processed_ids -> raise DuplicateParcelError(...)
        # TODO: передать parcel в storage[parcel.locker_id].add_parcel(parcel)
        # TODO: после успеха обновить processed_ids и accepted
        # TODO: LockerError сохранить в errors как (row, error_type, message)
        try:
            parcel = self.parse_parcel(row)
            if parcel.parcel_id in self.processed_ids:
                raise DuplicateParcelError('id уже существует!')
            self.storage[parcel.locker_id].add_parcel(parcel)
            self.accepted.append(parcel)
            self.processed_ids.add(parcel.parcel_id)
        except LockerError as e:
            self.errors.append((row, type(e).__name__, e))

        
    def load(self, rows):
        # TODO: вызвать submit(row) для каждой строки
        for el in rows:
            self.submit(el)

    def fullest_locker(self):
        # TODO: найти Locker с максимальной долей заполнения used_volume() / capacity_l
        # TODO: вернуть tuple(locker_id, used_volume, capacity_l)
        m = 0
        lck = None
        for key, val in self.storage.items():
            if (val.used_volume() / val.capacity_l) > m:
                m = (val.used_volume() / val.capacity_l)
                lck = val
        return (lck.locker_id, lck.used_volume(), lck.capacity_l)


    def find_parcel(self, parcel_id):
        # TODO: пройтись по всем локерам и их посылкам
        # TODO: если parcel.parcel_id совпал -> вернуть объект Parcel
        # TODO: иначе вернуть None
        for el in self.storage.values():
            for prc in el.parcels:
                if prc.parcel_id == parcel_id:
                    return prc
        return None

    def client_volume(self, client):
        # TODO: посчитать суммарный volume_l всех принятых посылок клиента
        m = 0
        for el in self.accepted:
            if el.client == client:
                m += el.volume_l
        return m


service = LockerService(locker_specs)

# TODO: загрузить rows через service.load(rows)
# TODO: вывести принятые посылки
# TODO: вывести ошибки
# TODO: вывести по каждому локеру список посылок и свободный объём
# TODO: вывести fullest_locker()
# TODO: вывести find_parcel('PX-106')
# TODO: вывести client_volume('Bob')

service.load(rows)
print(f'принятые: {service.accepted}')
print('--------------------------')
print(f'ошибки: {service.errors}')
obj = {}
for id, val in service.storage.items():
    obj.setdefault(id, {'lists_of_parcels': [], 'free_volume': 0})
    obj[id]['lists_of_parcels'].extend(val.parcels)
    obj[id]['free_volume'] = val.free_volume()

print(obj)
print(f'Самый заполненый locker: {service.fullest_locker()}')
print(f'посылка PX-106: {service.find_parcel('PX-106')}')
print(service.client_volume('Bob'))




принятые: [Parcel(volume_l=2, parcel_id='PX-100', locker_id='COLD-1', client='Alice', mode='cold'), Parcel(volume_l=3, parcel_id='PX-101', locker_id='DRY-1', client='Bob', mode='dry'), Parcel(volume_l=4, parcel_id='PX-106', locker_id='COLD-2', client='Ira', mode='cold')]
--------------------------
ошибки: [('PX-102|COLD-1|Kira|5|dry', 'StorageModeError', StorageModeError('режимы должны совпадать!')), ('PX-103|HOT-9|Max|1|dry', 'LockerNotFoundError', LockerNotFoundError('не удается найти такой locker_id!')), ('PX-104|COLD-2|Eva|0|cold', 'VolumeError', VolumeError('volume_raw должно быть больше 0!')), ('PX-101|DRY-1|Bob|2|dry', 'DuplicateParcelError', DuplicateParcelError('id уже существует!')), ('PX-105|COLD-1|Lena|7|cold', 'CapacityLimitError', CapacityLimitError('volume_l больше free_volume!'))]
{'COLD-1': {'lists_of_parcels': [Parcel(volume_l=2, parcel_id='PX-100', locker_id='COLD-1', client='Alice', mode='cold')], 'free_volume': 6}, 'COLD-2': {'lists_of_parcels': [Parcel(volume_l=4,